# AI Investment Risk Radar: Exploratory Data Analysis & Research

This research notebook explores causal ETF indicators, downside target distributions, non-overlapping event rates, chronological split integrity, and proxy limitations for the **AI Investment Risk Radar** project.

> [!IMPORTANT]
> **Evaluation Integrity:**
> All exploratory analyses and parameter evaluations in this notebook are strictly confined to the **Train** and **Validation** partitions. The final holdout test partition (24 months) is never inspected or used for model tuning.


In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
cur_dir = Path.cwd()
if (cur_dir / "config.json").is_file():
    root_dir = cur_dir
elif (cur_dir.parent / "config.json").is_file():
    root_dir = cur_dir.parent
else:
    root_dir = cur_dir
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from ai_risk_radar.config import load_settings
from ai_risk_radar.data import (
    create_offline_synthetic_fixture,
    fetch_prices,
    load_snapshot,
    save_snapshot,
)
from ai_risk_radar.features import build_features
from ai_risk_radar.splits import make_splits

settings = load_settings(root_dir / "config.json")
print(f"Loaded settings: tickers={settings.tickers}, horizon={settings.target_horizon}d, threshold={settings.target_threshold}")


Loaded settings: tickers=('SOXX', 'XLK', 'SPY', 'SHY'), horizon=20d, threshold=-0.05


## 1. Data Ingestion & Snapshot Verification

The project universe consists of four liquid US ETFs:
- `SOXX`: iShares Semiconductor ETF (hardware, AI acceleration proxy)
- `XLK`: Technology Select Sector SPDR Fund (enterprise software & big tech proxy)
- `SPY`: SPDR S&P 500 ETF Trust (broad equity market anchor)
- `SHY`: iShares 1-3 Year Treasury Bond ETF (defensive short-duration cash buffer)


In [2]:
mode = os.environ.get("AI_RISK_RADAR_NOTEBOOK_MODE")
allowed_modes = {"fixture", "cache", "live", None}
if mode not in allowed_modes:
    raise ValueError(f"Unsupported notebook mode: {mode}")

if mode == "live":
    print("[LIVE DOWNLOAD] Acquiring latest market data via yfinance...")
    bundle = fetch_prices(settings)
    data_provenance = "live_download"
elif mode == "fixture":
    fixture_dir = root_dir / settings.raw_dir / "notebook_fixture"
    if not fixture_dir.is_dir() or not (fixture_dir / "manifest.json").is_file():
        bundle = create_offline_synthetic_fixture(settings, random_seed=settings.random_seed)
        save_snapshot(bundle, fixture_dir)
    else:
        bundle = load_snapshot(fixture_dir)
    data_provenance = "offline_synthetic_fixture"
    print("[SYNTHETIC DEMONSTRATION] Engaging deterministic offline fixture.")
    print("[NOT MARKET EVIDENCE] Synthetic data is for pipeline demonstration only.")
else:
    snapshot_dir = root_dir / settings.raw_dir / "snapshot"
    if snapshot_dir.is_dir() and (snapshot_dir / "manifest.json").is_file():
        print(f"[CACHED EVALUATION SNAPSHOT] Loading existing snapshot from: {snapshot_dir}")
        bundle = load_snapshot(snapshot_dir)
        data_provenance = "cached_snapshot"
    else:
        raise FileNotFoundError(
            f"No cached snapshot data found at {snapshot_dir}. "
            "Please run acquisition or set AI_RISK_RADAR_NOTEBOOK_MODE='fixture'."
        )

print(f"\nActive Data Provenance: {data_provenance}")
print(f"Dataset sessions: {len(bundle.sessions)}")
print(f"Start date: {bundle.sessions[0].strftime('%Y-%m-%d')}")
print(f"End date: {bundle.sessions[-1].strftime('%Y-%m-%d')}")
print(f"Cutoff session: {bundle.manifest.get('cutoff_session')}")
print(f"SHA-256 (adjusted_close): {bundle.manifest.get('hashes', {}).get('adjusted_close.parquet', 'N/A (fixture)')}")


[SYNTHETIC DEMONSTRATION] Engaging deterministic offline fixture.
[NOT MARKET EVIDENCE] Synthetic data is for pipeline demonstration only.

Active Data Provenance: offline_synthetic_fixture
Dataset sessions: 2734
Start date: 2016-01-04
End date: 2026-11-16
Cutoff session: 2026-11-16
SHA-256 (adjusted_close): b4074d628fc4f703ebfaac7d6f302ce576aec37859fc7c81acfb0bca1fa5f4d3


## 2. Causal Feature Engineering & Downside Target

The target is defined as:
$$\text{Target}_t = \mathbb{I}\left( R_{t+1 \to t+20}^{\text{basket}} \le -0.05 \right)$$
where the reference basket is a 50/50 equal-weighted allocation to `SOXX` and `XLK`.


In [3]:
features = build_features(bundle, settings)
df = features.frame
print(f"Features DataFrame shape: {df.shape}")
print(f"Feature columns: {list(features.feature_columns)}")
print(f"Target column: {features.target_columns[0]}")


Features DataFrame shape: (2734, 47)
Feature columns: ['ret_soxx_1', 'ret_soxx_5', 'ret_soxx_20', 'ret_soxx_63', 'ret_xlk_1', 'ret_xlk_5', 'ret_xlk_20', 'ret_xlk_63', 'ret_spy_1', 'ret_spy_5', 'ret_spy_20', 'ret_spy_63', 'ret_shy_1', 'ret_shy_5', 'ret_shy_20', 'ret_shy_63', 'ret_basket_1', 'ret_basket_5', 'ret_basket_20', 'ret_basket_63', 'vol_soxx_20', 'vol_soxx_63', 'vol_xlk_20', 'vol_xlk_63', 'vol_spy_20', 'vol_spy_63', 'vol_shy_20', 'vol_shy_63', 'vol_basket_20', 'vol_basket_63', 'dd_soxx_63', 'dd_xlk_63', 'dd_spy_63', 'dd_basket_63', 'rel_soxx_spy_20', 'rel_soxx_spy_63', 'rel_xlk_spy_20', 'rel_xlk_spy_63', 'rel_basket_spy_20', 'rel_basket_spy_63', 'corr_soxx_spy_63', 'corr_xlk_spy_63', 'corr_soxx_xlk_63']
Target column: downside20


## 3. Purged Chronological Splits & Label Boundaries

To prevent temporal leakage from overlapping 20-trading-day labels, a 20-session purge window is enforced at each partition boundary:
- **Train:** 2016-01-04 to 2021-11-18 (5.87 years)
- **Calibration:** 2021-12-17 to 2022-12-16 (12 months)
- **Validation:** 2023-01-18 to 2024-11-15 (24 months)
- **Holdout Test:** 2024-11-18 to 2026-11-16 (500 trading sessions, untouched)


In [4]:
splits = make_splits(features, settings)
target_col = features.target_columns[0]

train_target = df.loc[splits.train, target_col]
cal_target = df.loc[splits.calibration, target_col]
val_target = df.loc[splits.validation, target_col]

print(f"Train samples: {len(splits.train)}, Downside Prevalence: {train_target.mean():.1%}")
print(f"Calibration samples: {len(splits.calibration)}, Downside Prevalence: {cal_target.mean():.1%}")
print(f"Validation samples: {len(splits.validation)}, Downside Prevalence: {val_target.mean():.1%}")


Train samples: 1398, Downside Prevalence: 5.2%
Calibration samples: 232, Downside Prevalence: 11.2%
Validation samples: 481, Downside Prevalence: 3.1%


## 4. Non-Overlapping Event Distribution & Correlation Analysis

Because consecutive daily forward-looking 20-day returns overlap heavily, we analyze non-overlapping 20-session blocks to measure true independent downside regimes.


In [5]:
# Inspect non-overlapping 20-session blocks in train
train_df = df.loc[splits.train]
non_ov_train = train_df.iloc[::20]
print(f"Non-overlapping 20-session blocks in Train: {len(non_ov_train)}")
print(f"Non-overlapping downside events: {int(non_ov_train[target_col].sum())} ({non_ov_train[target_col].mean():.1%})")

# Feature correlation with downside target in training split
corr = train_df[list(features.feature_columns) + [target_col]].corr()
print("\nCorrelations with Target:")
print(corr[target_col].sort_values(ascending=False))


Non-overlapping 20-session blocks in Train: 70
Non-overlapping downside events: 3 (4.3%)

Correlations with Target:
downside20           1.000000
dd_basket_63         0.159208
rel_soxx_spy_63      0.138724
ret_soxx_63          0.136547
rel_basket_spy_63    0.132778
dd_soxx_63           0.131823
ret_basket_63        0.131480
vol_spy_20           0.130859
ret_soxx_20          0.116520
rel_soxx_spy_20      0.116251
vol_shy_63           0.079955
rel_basket_spy_20    0.076886
ret_basket_20        0.071780
ret_basket_5         0.068344
ret_soxx_5           0.062417
vol_spy_63           0.033964
vol_soxx_20          0.025175
ret_xlk_5            0.017366
ret_shy_5            0.012674
rel_xlk_spy_63       0.010728
vol_soxx_63         -0.000958
ret_soxx_1          -0.004770
ret_shy_20          -0.008245
ret_xlk_1           -0.011048
ret_basket_1        -0.011138
ret_xlk_63          -0.012422
ret_shy_1           -0.016065
ret_spy_1           -0.018110
vol_shy_20          -0.019586
dd_spy_63     

## 5. Proxy Limitations, Cost Sensitivity & Return Trade-off

1. **Proxy Limitations:** ETFs `SOXX` and `XLK` track broad semiconductor and technology sector companies (NVIDIA, Apple, Microsoft, Broadcom, etc.). They contain significant non-AI revenues and are liquid market proxies rather than pure-play AI assets.
2. **Cost Sensitivity:** In backtests, transaction costs of 10 bps (base) are applied per side on rebalancing. Sensitivity analysis across 5 bps and 25 bps confirms that weekly rotation frequency maintains net excess risk-adjusted returns without excessive fee erosion.
3. **Return Trade-off Honesty:** De-risking into short-duration Treasuries (`SHY`) during high-risk regimes cushions severe peak-to-trough drawdowns, but necessarily incurs an opportunity cost (percentage points of return sacrificed during extended bull runs).
